
# Fase 3: Codificación y Preparación de Datos

En esta fase del proyecto se realiza la **transformación de variables** para preparar el dataset para modelos de *Machine Learning*.  
El objetivo es convertir todas las variables en formato **numérico**, reducir la **dimensionalidad** y asegurar que los datos estén listos para el entrenamiento de modelos.

## Objetivos de esta fase

1. Analizar variables categóricas del dataset.
2. Reducir categorías poco frecuentes (especialmente en variables como `City` y `Degree`).
3. Aplicar técnicas de codificación:
   - **OneHotEncoder** para variables categóricas.
   - **OrdinalEncoder** para variables binarias.
   - **StandardScaler** para variables numéricas.
4. Construir un **pipeline de transformación** usando `ColumnTransformer`.
5. Generar un dataset final listo para modelado.

Dataset utilizado: **Student Depression Dataset (versión limpia)**.


In [33]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

In [34]:
import numpy as np
import pandas as pd
from src.carga import cargar_csv
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from src.eda_utils import *

In [35]:
df = cargar_csv("..\data\\processed\Student_Depression_Dataset_Limpio.csv")

In [36]:
df_codificado = df.copy()

In [37]:
df_codificado_2 = df.copy()

In [38]:
df_codificado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 15 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27817 non-null  int64  
 1   Gender                                 27817 non-null  object 
 2   Age                                    27817 non-null  float64
 3   City                                   27817 non-null  object 
 4   Academic Pressure                      27817 non-null  float64
 5   CGPA                                   27817 non-null  float64
 6   Study Satisfaction                     27817 non-null  float64
 7   Sleep Duration                         27817 non-null  object 
 8   Dietary Habits                         27817 non-null  object 
 9   Degree                                 27817 non-null  object 
 10  Have you ever had suicidal thoughts ?  27817 non-null  object 
 11  Wo

| Método         | Qué hace                                                   |
| -------------- | -----------------------------------------------------------|
| LabelEncoder   | convierte categorías en números                            |
| OneHotEncoder  | crea columnas binarias                                     |
| OrdinalEncoder | asigna números pero pensado para variables ordinales       |
| StandardScaler | normaliza datos númericos y los asigna en una misma escala |

In [39]:
df_codificado.select_dtypes(include='object').nunique()

Gender                                    2
City                                     30
Sleep Duration                            4
Dietary Habits                            3
Degree                                   28
Have you ever had suicidal thoughts ?     2
Family History of Mental Illness          2
dtype: int64

In [40]:
# Columnas que tienen muchas categorías y que queremos simplificar
# En este caso se seleccionan 'City' y 'Degree' porque suelen tener muchos valores distintos
cols = ['City', 'Degree']

# Recorremos cada columna de la lista
for col in cols:
    
    # 1. Contamos cuántas veces aparece cada categoría
    # value_counts() devuelve las categorías ordenadas por frecuencia
    # nlargest(10) selecciona solo las 10 más frecuentes
    top = df_codificado[col].value_counts().nlargest(10).index
    
    # 2. Reemplazamos las categorías poco frecuentes
    # Si el valor está dentro de las 10 más comunes se mantiene
    # Si no, se reemplaza por la categoría "Other"
    df_codificado[col] = df_codificado[col].apply(lambda x: x if x in top else 'Other')

In [41]:
df_codificado['City'].unique()

array(['Other', 'Srinagar', 'Thane', 'Kalyan', 'Kolkata', 'Lucknow',
       'Surat', 'Ludhiana', 'Agra', 'Hyderabad', 'Vasai-Virar'],
      dtype=object)

In [42]:
df_codificado['City'].nunique()

11


## Construcción del Pipeline de Transformación

Para preparar los datos se utiliza **ColumnTransformer**, lo que permite aplicar diferentes transformaciones según el tipo de variable.

### Transformaciones aplicadas

**Variables categóricas**
- Se utiliza `OneHotEncoder`.
- Se aplica `drop='first'` para eliminar una categoría redundante y evitar multicolinealidad.

**Variable binaria**
- Se utiliza `OrdinalEncoder` para convertir respuestas tipo *Yes/No* en valores numéricos.

**Variables numéricas**
- Se utiliza `StandardScaler` para estandarizar los datos.
- Esto transforma las variables para que tengan:
  - Media = 0
  - Desviación estándar = 1

Esto mejora el rendimiento de varios algoritmos de Machine Learning.


In [43]:
df_codificado['City'].value_counts()

City
Other          15623
Kalyan          1569
Srinagar        1369
Hyderabad       1339
Vasai-Virar     1287
Lucknow         1152
Thane           1139
Ludhiana        1107
Agra            1091
Surat           1077
Kolkata         1064
Name: count, dtype: int64

In [ ]:
pipeline = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
         [
             'Gender',
             'Sleep Duration',
             'Dietary Habits',
             'Degree',
             'Family History of Mental Illness'
         ]),

        ('bin', OrdinalEncoder(),
         [
             'Have you ever had suicidal thoughts ?'
         ]),

        ('num', StandardScaler(),
         [
             'Age',
             'Academic Pressure',
             'CGPA',
             'Study Satisfaction',
             'Work/Study Hours',
             'Financial Stress'
         ])
    ]
)

X = df.drop(columns=['Depression','id','City'])

x_transf = pipeline.fit_transform(X)

df_transf = pd.DataFrame(
    x_transf,
    columns=pipeline.get_feature_names_out()
)

df_transf

ValueError: A given column is not a column of the dataframe

In [49]:
pipeline = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
         [
             'Gender',
             'Sleep Duration',
             'Dietary Habits',
             'Degree',
             'Family History of Mental Illness'
         ]),

        ('bin', OrdinalEncoder(),
         [
             'Have you ever had suicidal thoughts ?'
         ]),

        ('num', StandardScaler(),
         [
             'Age',
             'Academic Pressure',
             'CGPA',
             'Study Satisfaction',
             'Work/Study Hours',
             'Financial Stress'
         ])
    ]
)

X = df_codificado_2.drop(columns=['Depression','id', 'City'])

x_transf_2 = pipeline.fit_transform(X)

df_transf_2 = pd.DataFrame(
    x_transf_2,
    columns=pipeline.get_feature_names_out()
)

df_transf_2

,cat__Gender_Male,cat__Sleep Duration_7-8 hours,cat__Sleep Duration_Less than 5 hours,cat__Sleep Duration_More than 8 hours,cat__Dietary Habits_Moderate,cat__Dietary Habits_Unhealthy,cat__Degree_B.Com,cat__Degree_B.Ed,cat__Degree_B.Pharm,cat__Degree_B.Tech,...,cat__Degree_Others,cat__Degree_PhD,cat__Family History of Mental Illness_Yes,bin__Have you ever had suicidal thoughts ?,num__Age,num__Academic Pressure,num__CGPA,num__Study Satisfaction,num__Work/Study Hours,num__Financial Stress
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.476593,1.345273,0.895400,-0.694638,-1.122130,-1.489063
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,-0.370683,-0.827109,-1.200717,1.510988,-1.122130,-0.793223
2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.066087,-0.102982,-0.429182,1.510988,0.496561,-1.489063
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.450328,-0.102982,-1.412377,-0.694638,-0.852348,1.294297
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-0.165430,0.621145,0.321870,0.040570,-1.661693,-1.489063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27812,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.245076,1.345273,-1.303133,1.510988,-0.043003,-1.489063
27813,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.245076,-0.827109,1.188993,0.040570,-1.931475,-0.097383
27814,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.066087,-0.102982,-0.715947,0.775779,1.305906,-0.793223
27815,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-1.602200,1.345273,-0.531598,-0.694638,0.766342,1.294297


In [46]:
for col in pipeline.get_feature_names_out():
    print(col)

cat__Gender_Male
cat__Sleep Duration_7-8 hours
cat__Sleep Duration_Less than 5 hours
cat__Sleep Duration_More than 8 hours
cat__Dietary Habits_Moderate
cat__Dietary Habits_Unhealthy
cat__Degree_B.Com
cat__Degree_B.Ed
cat__Degree_B.Pharm
cat__Degree_B.Tech
cat__Degree_BA
cat__Degree_BBA
cat__Degree_BCA
cat__Degree_BE
cat__Degree_BHM
cat__Degree_BSc
cat__Degree_Class 12
cat__Degree_LLB
cat__Degree_LLM
cat__Degree_M.Com
cat__Degree_M.Ed
cat__Degree_M.Pharm
cat__Degree_M.Tech
cat__Degree_MA
cat__Degree_MBA
cat__Degree_MBBS
cat__Degree_MCA
cat__Degree_MD
cat__Degree_ME
cat__Degree_MHM
cat__Degree_MSc
cat__Degree_Others
cat__Degree_PhD
cat__Family History of Mental Illness_Yes
bin__Have you ever had suicidal thoughts ?
num__Age
num__Academic Pressure
num__CGPA
num__Study Satisfaction
num__Work/Study Hours
num__Financial Stress


In [47]:
print(x_transf.shape)
print(len(pipeline.get_feature_names_out()))

(27817, 41)
41


In [48]:
df_transf.to_csv("..\data\processed\Student_Depression_Dataset_codificado.csv", index=False)